# Test Stage 12 — Replacing the two checks that only work on an emulator

Stage 10 decides two things about every point in the feedback atlas, and **both decisions are
impossible on a real coil**:

| Decision | How Stage 10 makes it | Why it cannot survive |
|---|---|---|
| Is this point oscillating? | Switch the coil's thermal noise **off** and see whether the signal is still there | Real thermal noise has no off switch |
| Is the loop still linear here? | Drop that noise by 12 dB and require the resonance to shrink by 16× | Real thermal noise is not adjustable either |

This stage builds a replacement for each, and checks it against the version that works.

### Why this must be done now, before the coil is connected

This is a one-way door. The moment the resonator is disabled and a real coil is cabled,
**there is no way left to check an answer against a known one.** The replacements can only be
trusted if they are compared against the originals, and the originals only run against the
emulator. Leaving this until the coil is attached means shipping two unchecked decisions
straight into the physics.

### What comes out of it

Two numbers — a threshold for each replacement — and the evidence that they classify the atlas
the same way the originals do.

---

### Simulation or the real trap?

**This stage is emulation-only by design, and that is the point.** It is a calibration of the
real-coil methods against the emulated ones. It never runs on the real coil; its *output* does.

## What you need

| | |
|---|---|
| Stages 1–10 | Passing |
| Stage 0 | Passing — this stage trusts the fitting underneath it |
| Bitstream | The one with the resonator and the noise sources |
| Time | About four minutes |
| Cables | **None.** Everything is internal |

## Setup — drivers, capture and the sweep (the Stage 10 set, unchanged)

In [ ]:
import os, mmap, struct, time
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

FS_CLK = 125e6
F0     = 40.0e6          # the coil being emulated: ~40 MHz
Q_SET  = 180.0           # ~180 at room temperature
SETTLE = 0.02
N_AVG  = 32

# =============================================================================
#  Environment shims. This board carries an older numpy and matplotlib than a
#  current desktop, and two conveniences used below arrived later than what is
#  installed here: nan_to_num's `nan=` keyword needs numpy >= 1.17, and
#  Axes.set_facecolor needs matplotlib >= 2.0. Provided once, rather than as
#  version tests scattered through the analysis cells.
# =============================================================================
def nan_fill(a, value):
    """np.nan_to_num(a, nan=value), for numpy versions without the keyword."""
    out = np.array(a, dtype=float, copy=True)
    out[~np.isfinite(out)] = value
    return out


def set_bg(ax, colour):
    """Axes background colour, across the matplotlib 1.x -> 2.x rename."""
    if hasattr(ax, 'set_facecolor'):
        ax.set_facecolor(colour)
    else:
        ax.set_axis_bgcolor(colour)


# =============================================================================
#  Drivers - the Stage 8 set, with the resonator gaining its two noise knobs.
#  CRITICAL RULE throughout: one 32-bit register access at a time.
# =============================================================================
class Bus(object):
    def __init__(self, base, span=0x1000):
        self.fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
        self.m  = mmap.mmap(self.fd, span, offset=base)
    def w(self, off, val):
        self.m[off:off + 4] = struct.pack('<I', val & 0xFFFFFFFF)
    def r(self, off):
        return struct.unpack('<I', self.m[off:off + 4])[0]

def s32(v):
    return v - (1 << 32) if v & 0x80000000 else v


class Acq(Bus):
    """Acquisition system as narrowband voltmeter (PEEK at DC) and as
    recorder (capture to memory - defined in the next cell)."""
    def __init__(self):
        Bus.__init__(self, 0x40600000)
        self.dec = 10
    def start(self, dec_log2=10):
        self.dec = dec_log2
        self.w(0x00, 0)
        self.w(0x08, dec_log2)
        self.w(0x0C, 0)
        self.w(0x00, 1 | 2)
        self.w(0x00, 1)
    def tune(self, f):
        self.w(0x04, int(round(f / FS_CLK * (1 << 32))) & 0xFFFFFFFF)
    def measure(self, navg=N_AVG):
        zi = 0.0; zq = 0.0
        for _ in range(navg):
            zi += s32(self.r(0x18))
            zq += s32(self.r(0x1C))
        return complex(zi / navg, zq / navg)
    def f_out(self):
        return FS_CLK / (1 << self.dec)


class Resonator(Bus):
    def __init__(self):
        Bus.__init__(self, 0x40400000)
        self.f0_q = None; self.q_q = None
    def config(self, f0, Q, delay=15, oshift=7):
        w0 = 2.0 * np.pi * f0 / FS_CLK
        r  = 1.0 - np.pi * (f0 / FS_CLK) / Q
        ci = int(round(r * np.cos(w0) * 131072))
        cs = int(round(r * np.sin(w0) * 131072))
        self.w(0x04, ci & 0x3FFFF)
        self.w(0x08, cs & 0x3FFFF)
        self.w(0x0C, delay)
        self.w(0x10, oshift)
        rq        = np.hypot(ci, cs) / 131072.0
        self.f0_q = np.arctan2(cs, ci) / (2.0 * np.pi) * FS_CLK
        self.q_q  = np.pi * (self.f0_q / FS_CLK) / (1.0 - rq)
    def shift_f0(self, df, Q=None):
        """Retune the emulated coil by df hertz - the drift the servo fights.
        One coefficient rewrite; Q is preserved."""
        self.config(self.f0_q + df, Q if Q is not None else self.q_q,
                    delay=15, oshift=7)
    # --- the two noise knobs (registers added for this stage) -------------
    def noise_tank(self, shift=None):
        """Johnson noise INTO the loop -> Lorentzian bump. None = off.
        Each shift step is 6 dB down from raw (~9460 counts rms)."""
        self.w(0x20, 0 if shift is None else (0x100 | (shift & 0x1F)))
    def noise_out(self, shift=None):
        """Amplifier noise AFTER the loop -> flat floor. None = off."""
        self.w(0x24, 0 if shift is None else (0x100 | (shift & 0x1F)))
    def enable(self):  self.w(0x00, 1)
    def disable(self): self.w(0x00, 0)
    def clear(self):   self.w(0x00, 3)
    def clipped(self): return bool(self.r(0x14) & 1)
    def peek(self):    return s32(self.r(0x18)), s32(self.r(0x1C))


class PhaseShift(Bus):
    LATENCY = 2
    COEF_FS = 16384.0                 # coefficient full scale -> gain LSB
    def __init__(self):
        Bus.__init__(self, 0x40300000)
    def config_delay(self, f):
        best = None
        for k in range(0, 9):
            D = FS_CLK / (4.0 * f) + k * FS_CLK / f
            if D < 1.0 or D > 1000:
                continue
            frac  = D - np.floor(D)
            score = min(frac, 1.0 - frac)
            if best is None or score < best[0]:
                best = (score, int(np.floor(D)), frac)
        self.w(0x08, best[1])
        self.w(0x0C, int(round(best[2] * 16384)))
        self.w(0x10, 0)
        return best[1], best[2]
    def set(self, A, phi_deg, f=F0):
        phi = phi_deg - 360.0 * self.LATENCY * f / FS_CLK
        self.w(0x00, int(round(A * np.cos(np.radians(phi)) * self.COEF_FS)) & 0xFFFF)
        self.w(0x04, int(round(A * np.sin(np.radians(phi)) * self.COEF_FS)) & 0xFFFF)
    def gain_lsb(self, phi_deg, f=F0):
        """Smallest change in A this actuator can express at this phase. The
        two coefficients quantise independently, so the step in |A| is set by
        the LARGER of the two projections. Step 5c compares this against the
        width of the band a software loop would need to steer within."""
        phi = np.radians(phi_deg - 360.0 * self.LATENCY * f / FS_CLK)
        return 1.0 / (self.COEF_FS * max(abs(np.cos(phi)), abs(np.sin(phi))))
    def off(self):
        self.set(0.0, 0.0)


class Generator(Bus):
    def __init__(self):
        Bus.__init__(self, 0x40200000, span=0x20000)
    def load_sine(self):
        for i in range(16384):
            v = int(round(8191.0 * np.sin(2.0 * np.pi * i / 16384.0)))
            self.w(0x10000 + 4 * i, v & 0x3FFF)
        self.w(0x08, (16384 << 16) - 1)
        self.w(0x0C, 0)
    def amp(self, frac):
        self.w(0x04, int(round(frac * 8192)) & 0x3FFF)
    def freq(self, f):
        self.w(0x10, int(round(f / FS_CLK * (1 << 30))))
    def run(self):
        self.w(0x00, 0x40)
        self.w(0x00, 0x11)
    def stop(self):
        self.amp(0.0)
        self.w(0x00, 0xC0)


acq, reso, ps, gen = Acq(), Resonator(), PhaseShift(), Generator()
print('drivers ready  (numpy %s, matplotlib %s)'
      % (np.__version__, matplotlib.__version__))

In [ ]:
# =============================================================================
#  Memory capture - the Stage 6/7 system, condensed. Noise SPECTRA need time
#  records, and time records need memory; the PEEK voltmeter cannot see a
#  spectrum. Same safeguards as Stage 7: region from the device tree, stop
#  address inclusive, poison pre-fill, write-error and poison checks that
#  RAISE rather than return plausible garbage. (Bitstreams from the Stage 8
#  rebuild onward carry the trusted badge, so TrustZone no longer matters -
#  but the checks stay, because they cost nothing and catch everything.)
# =============================================================================
def _be32(b, i=0):
    return struct.unpack('>I', b[i * 4:i * 4 + 4])[0]

def _dt_read(path):
    for root in ('/sys/firmware/devicetree/base', '/proc/device-tree'):
        try:
            with open(os.path.join(root, path), 'rb') as fh:
                return fh.read()
        except (IOError, OSError):
            pass
    return None

blob = _dt_read('memory/reg') or _dt_read('memory@0/reg')
assert blob and len(blob) >= 8, 'cannot read the device tree memory node'
declared_end = _be32(blob, 0) + _be32(blob, 1)
assert declared_end < 512 * 1024**2, 'no memory above the declared end - see Stage 6 Step 1'
MEM_START = (declared_end + 0xFFF) & ~0xFFF
MEM_STOP  = MEM_START + (0x00800000 & ~7) - 8       # LAST address, inclusive
POISON    = 0xDEADBEEF
print('capture region 0x%08X .. 0x%08X inclusive' % (MEM_START, MEM_STOP + 7))


CAPTURE_TRACE = None      # set to a path (e.g. '/tmp/capture.log') to log every
                          # capture to disk before it runs


def _trace(msg):
    """Append to the trace file and force it to disk.

    A dead kernel takes buffered stdout with it, which is why a crash inside
    capture() leaves the notebook showing nothing at all. This survives,
    so the last line in the file names the call that killed it."""
    if not CAPTURE_TRACE:
        return
    with open(CAPTURE_TRACE, 'a') as fh:
        fh.write('%.3f %s\n' % (time.time(), msg))
        fh.flush()
        os.fsync(fh.fileno())


def _poison(nbytes):
    fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
    mm = mmap.mmap(fd, nbytes, offset=MEM_START)
    try:
        mm[0:nbytes] = np.full(nbytes // 4, POISON, dtype='<u4').tobytes()
    finally:
        mm.close(); os.close(fd)


def capture(f_lo, dec_log2, n_samples, settle=SETTLE):
    """Record n_samples of the complex stream; raises on any transport fault."""
    _trace('capture f=%.1f dec=%d n=%d' % (f_lo, dec_log2, n_samples))
    _poison(n_samples * 8)
    acq.w(0x00, 0)
    acq.w(0x08, dec_log2); acq.dec = dec_log2
    acq.w(0x0C, 0)
    acq.tune(f_lo)
    acq.w(0x28, MEM_START)
    acq.w(0x2C, MEM_STOP)
    # Read the bounds back BEFORE arming. The writer is a DMA master: if the
    # stop bound did not take - wrong bitstream, unimplemented register, a
    # write that silently went nowhere - it runs from MEM_START upward through
    # whatever follows, and what follows is the memory Linux is using. That is
    # not an exception with a traceback to read; it is a dead kernel. Two
    # register reads cost microseconds and turn the worst failure mode in this
    # notebook into a sentence.
    lo_rb, hi_rb = acq.r(0x28), acq.r(0x2C)
    if lo_rb != MEM_START or hi_rb != MEM_STOP:
        acq.w(0x00, 0)                              # ensure nothing is armed
        raise RuntimeError(
            'capture bounds did not take: wrote 0x%08X..0x%08X, read back '
            '0x%08X..0x%08X. Refusing to arm an unbounded DMA writer.'
            % (MEM_START, MEM_STOP, lo_rb, hi_rb))
    if n_samples * 8 > (MEM_STOP + 8 - MEM_START):
        raise ValueError('record of %d samples (%d bytes) exceeds the %d-byte '
                         'capture region'
                         % (n_samples, n_samples * 8, MEM_STOP + 8 - MEM_START))
    # SETTLE FIRST, RECORD SECOND. Arming immediately and sleeping afterwards
    # would let the record's opening milliseconds hold the filter pipeline still
    # flushing the PREVIOUS tuning - which in a sweep poisons each point with its
    # predecessor's power, most visibly the first point, and can drag a fit wide
    # by tens of percent from the far end of the grid. Instead the chain
    # free-runs at the new tuning while the pipeline flushes, and only then does
    # the writer start: sample 0 of every record is already settled.
    acq.w(0x00, 1 | 2)                              # run at the new tuning...
    acq.w(0x00, 1)
    time.sleep(settle)                              # ...flush the pipeline...
    acq.w(0x00, 1 | 16 | 128)                       # ...then arm, fresh writer
    acq.w(0x00, 1 | 16)
    # WAIT FOR THE WRITER; DO NOT GUESS AT IT.
    #
    # This used to be one sleep of "as long as the record should take, plus
    # 5 ms", after which the writer was disarmed and memory read. Two things
    # were wrong with that, and both showed up on 2026-08-04 as an intermittent
    # 'unwritten words in the record' on captures that were perfectly healthy:
    #
    #  1. DMA_COUNT (0x34) was not consulted until AFTER the decision to stop
    #     had already been made. The sleep was a guess with no verification.
    #  2. DMA_COUNT reports samples HANDED TO the memory writer, not samples
    #     that have landed in DDR. The writer bursts, so the last burst can
    #     still be in flight when the count already reads complete.
    #
    # A fixed 5 ms of margin is also the wrong shape: it is 30% of a short
    # record and 0.1% of a long one, so the failure is rare, load-dependent and
    # looks like a hardware fault. It is not - the record was read a few
    # hundred microseconds early.
    #
    # Poll until the record is complete, then let the last burst land.
    t_record = n_samples * (1 << dec_log2) / FS_CLK
    deadline = time.time() + t_record + 0.5         # generous, not 5 ms
    while int(acq.r(0x34)) < n_samples and time.time() < deadline:
        time.sleep(0.001)
    acq.w(0x00, 1)                                  # disarm
    time.sleep(0.002)                               # last burst reaches DDR

    st  = acq.r(0x38)
    if st & 8:
        raise RuntimeError('memory refused writes - see Stage 6; run check_system.py')
    got = min(int(acq.r(0x34)), n_samples)
    # try/finally, because everything between the open and the close can raise:
    # mmap itself, the slice (SIGBUS on a bad physical address), an allocation
    # failure on a tight board. Without it each failure leaks a file descriptor
    # AND a mapping of /dev/mem - and Stage 10's scan loop swallows per-point
    # exceptions by design, so the leak would repeat silently across hundreds
    # of grid points until the process runs out of descriptors or address
    # space. That surfaces later as an unrelated-looking out-of-memory error.
    def _read_region(nbytes):
        fd = mm = None
        try:
            fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
            mm = mmap.mmap(fd, ((nbytes + 8191) // 4096) * 4096, offset=MEM_START)
            return mm[0:nbytes]
        finally:
            if mm is not None:
                mm.close()
            if fd is not None:
                os.close(fd)

    buf = _read_region(got * 8)
    u32 = np.frombuffer(buf, dtype='<u4')
    _limit = max(8, 0.001 * u32.size)
    if got and int(np.count_nonzero(u32 == POISON)) > _limit:
        # A word still inside the writer's burst buffer is not a lost word. If
        # it is genuinely lost it will still be poison in 20 ms; if it was
        # merely late it will not. Retrying once separates the two, which the
        # single-shot check could not: it reported a transport fault for both.
        time.sleep(0.02)
        buf = _read_region(got * 8)
        u32 = np.frombuffer(buf, dtype='<u4')
        n_bad = int(np.count_nonzero(u32 == POISON))
        if n_bad > _limit:
            raise RuntimeError(
                'unwritten words in the record - transport fault, see Stage 6 '
                '(%d of %d words still unwritten 20 ms after the writer '
                'reported the record complete)' % (n_bad, u32.size))
    raw = np.frombuffer(buf, dtype='<i4').astype(np.float64)
    acq.w(0x00, 1)                                  # back to free-running voltmeter
    _trace('   ok got=%d' % got)
    return raw[0::2] + 1j * raw[1::2]


def welch_psd(z, seg=2048):
    """Averaged power spectrum of a complex record - the standard way to see
    noise: many short transforms averaged beat one long one, because noise in
    a single transform is 100% uncertain no matter how long it is."""
    k = len(z) // seg
    w = np.hanning(seg)
    p = np.zeros(seg)
    for i in range(k):
        p += np.abs(np.fft.fft(z[i * seg:(i + 1) * seg] * w)) ** 2
    f = np.fft.fftfreq(seg, 1.0 / acq.f_out())
    order = np.argsort(f)
    return f[order], p[order] / max(k, 1)


def _lorentz_core(f, p, guard=0.05, minpts=3):
    """Centre, width, peak and BASELINE of a Lorentzian power curve.

    1/p is a parabola in f only for a pure Lorentzian. Any additive floor -
    detector noise, an amplifier pedestal - is absorbed into that parabola and
    inflates the fitted width. That is negligible where the peak towers over
    the floor and severe where it does not, because peak power falls as the
    square of the Q ratio: a floor worth 2% of the open-loop peak is worth 32%
    of a 4x-broadened one. Uncorrected bias, measured: +18% in linewidth at
    Q_eff/Q0 = 1, +26% at 0.5, +153% at 0.25.

    The floor is found by search, not from the wings. For each trial floor:
    subtract it, fit, rebuild the full Lorentzian-plus-floor, and keep the
    trial that reproduces the measured points best. A wing estimate needs
    points beyond ~1.5 linewidths, which a broad line does not have on this
    grid - exactly the case that needs correcting most. The search uses every
    point, so it works at any width. Two passes: coarse over [0, min(p)], then
    fine around the winner.

    Returns (f0, linewidth, peak_above_baseline, baseline).
    """
    f = np.asarray(f, dtype=float)
    p = np.asarray(p, dtype=float)
    pmax = float(np.max(p))
    u = f - f.mean()

    def _try(b):
        q = p - b
        keep = q > guard * q.max()
        if int(np.count_nonzero(keep)) < minpts:
            return None
        A, B, C = np.polyfit(u[keep], 1.0 / q[keep], 2, w=q[keep])
        if A <= 0.0:
            return None                    # parabola opens the wrong way
        u0 = -B / (2.0 * A)
        lw = 2.0 * np.sqrt(max(C / A - u0 ** 2, 1e-30))
        p0 = 1.0 / max(C - A * u0 ** 2, 1e-300)
        f0 = f.mean() + u0
        mod = p0 / (1.0 + (2.0 * (f - f0) / lw) ** 2) + b
        return float(np.sqrt(np.mean(((p - mod) / pmax) ** 2))), (f0, lw, p0, b)

    hi = 0.999 * float(np.min(p))
    best, best_b = _try(0.0), 0.0           # b = 0 is the uncorrected fit
    if hi > 0.0:
        for b in np.linspace(0.0, hi, 17):
            got = _try(b)
            if got is not None and (best is None or got[0] < best[0]):
                best, best_b = got, b
        w = hi / 16.0
        for b in np.linspace(max(best_b - w, 0.0), min(best_b + w, hi), 17):
            got = _try(b)
            if got is not None and (best is None or got[0] < best[0]):
                best, best_b = got, b
    if best is None:
        raise ValueError('fit window too narrow: %d points'
                         % int(np.count_nonzero(p > guard * p.max())))
    f0, lw, p0, base = best[1]
    return f0, lw, p0, base


def fit_lorentzian_psd(f, p, guard=0.05):
    """Centre, width from a power spectrum: 1/p is a parabola (Stage 8), after
    the noise floor has been taken out of p."""
    f0, lw, _, _ = _lorentz_core(f, p, guard=guard, minpts=3)
    return f0, lw

In [ ]:
# ============================ CONFIGURATION ==================================
EMULATION  = True    # True: digital resonator, known answers, PASS gates.
                     # False: the real coil - same scan, gates become measurements

TANK_SH    = 3       # tank (Johnson) noise level, 6 dB per step
OUT_SH     = 4       # amplifier-floor level (off during sweeps automatically)

# The gain axis must BRACKET the self-oscillation boundary, or there is no
# boundary in the atlas and every stability margin the designer prints is
# meaningless. K rises ~0.73 per unit A, so onset is near A = 1.4 (+3 dB).
GAINS_DB   = np.arange(-26.0, 6.01, 2.0)     # feedback gain axis (dB of A)
PHIS       = np.arange(0.0, 360.0, 15.0)     # feedback phase axis: 24 points
SWEEP_PTS  = 15      # frequencies per mini-sweep at each grid point
SWEEP_SPAN = 2.5     # mini-sweep half-width in open-loop linewidths
SWEEP_N    = 2048    # samples per frequency point (halved at deep decimations)
SETTLE     = 0.02

# A point is OSCILLATING if, with the noise drive removed, it still holds this
# fraction of its driven power. A ratio, so it needs no absolute floor - on a
# digital plant the undriven floor is exactly zero.
SUS_OSC    = 0.15
DRIVE_STEP = 2       # noise-shift steps (12 dB) used to test loop linearity
# Grid cost: 17 x 24 points x ~1 s. About 7 minutes.
# =============================================================================

# --- Step 0: right bitstream, plant configured, knob calibrated --------------
reso.w(0x20, 0x105)
assert (reso.r(0x20) & 0x1FF) == 0x105, 'noise registers absent - rebuild (see Stage 9 Step 0)'
reso.w(0x20, 0)

reso.config(F0, Q_SET)
ps.config_delay(F0); ps.off()
gen.load_sine(); gen.amp(0.0); gen.run()
reso.enable(); reso.clear()
acq.start(dec_log2=10); acq.tune(reso.f0_q)
time.sleep(0.2)
z_bg = acq.measure(64)
LW0  = reso.f0_q / reso.q_q


def coherence(z):
    """1 for pure noise, 0 for a pure tone; equals 1 - f^2 where f is the
    power fraction carried by a coherent tone. Reported as a diagnostic.
    Not used to classify points: for narrowband noise the effective sample
    count is small and this scatters ~30%, far too coarse to detect anything
    subtle."""
    p = np.abs(z) ** 2
    m = float(np.mean(p))
    return float(np.var(p)) / max(m * m, 1e-300)


def tone_fraction(R):
    return float(np.sqrt(max(0.0, 1.0 - R)))


def fit_lorentz3(f, p, guard=0.10):
    """Centre, width, peak and baseline of a Lorentzian power curve (the
    Stage 9 method, with the floor removed - see _lorentz_core in the cell
    above). Raises if the guard leaves too few points to determine a parabola;
    silence there is how a thin fit window becomes a huge Q.

    The peak is now the height ABOVE the baseline, so (pi/2)*p0*lw is the area
    of the bump alone - which is what T_eff wants. The baseline is returned so
    the caller can rebuild the full curve when testing the fit."""
    return _lorentz_core(f, p, guard=guard, minpts=5)


def _win_half(dec):
    """The receiver's listening half-width at decimation 2^dec."""
    return 0.27 * FS_CLK / (1 << dec)


def _dec_for(spacing):
    """Match the receiver to the grid: window ~ half the point spacing,
    rounded WIDER. A too-wide window smears the width, which the
    deconvolution below undoes exactly; a too-narrow one leaves gaps a narrow
    line can hide in, which no amount of fitting can undo."""
    d = int(np.floor(np.log2(0.27 * FS_CLK / (0.5 * spacing))))
    return int(np.clip(d, 8, 14))


def _model(fs, dec, f0, lw):
    """A width-lw Lorentzian at f0 seen through this receiver on this grid:
    the window-averaged (smeared) curve, in closed form."""
    W = _win_half(dec)
    return ((lw / 2.0) * (np.arctan(2.0 * (fs + W - f0) / lw)
                        - np.arctan(2.0 * (fs - W - f0) / lw)) / (2.0 * W))


def _deconv_lw(lw_m, f0, fs, dec):
    """Undo the window smear with no calibration constants: bisect the true
    width whose model, pushed through the SAME window, grid and fit,
    reproduces the measured width.

    Returns (lw_true, bracketed). `bracketed` means the search hit its lower
    bound - the correction wanted to be more than 4x and was truncated, which
    would inflate Q with no other symptom. The caller discards those."""
    def fwd(lw_t):
        # The guard in fit_lorentz3 protects fits to real DATA. Tripping it on
        # our own noise-free model just means the trial width is too narrow for
        # this grid to express, which is the same as "narrower than measured".
        try:
            return fit_lorentz3(fs, _model(fs, dec, f0, lw_t) + 1e-12)[1]
        except ValueError:
            return 0.0
    lo, hi = lw_m / 4.0, lw_m * 1.05
    if fwd(hi) < lw_m:
        return lw_m, False               # smear negligible at this geometry
    for _ in range(30):
        mid = 0.5 * (lo + hi)
        if fwd(mid) < lw_m:
            lo = mid
        else:
            hi = mid
    out = 0.5 * (lo + hi)
    return out, bool(out < 1.02 * lw_m / 4.0)


def noise_sweep(centre, pts=SWEEP_PTS):
    """Adaptive power sweep. Each pass matches the receiver to the grid, then
    reads the data before trusting any fit:
      - low contrast          -> structure wider than the span: widen, retry
      - one point owns the power -> narrower than the grid: zoom on it
      - otherwise fit, accepting only if the centre is inside the span, the
        width is resolvable here, and the curve goes through the points
    An accepted width is deconvolved from the receiver window. Five passes
    maximum; anything still unresolved returns nans - reported, never stored.

    Returns (f0, lw_true, p0, area, coh). Area uses the MEASURED (smeared)
    peak and width: window averaging conserves the integral, so that pair
    gives the true area directly, while Q needs the deconvolved width."""
    span = SWEEP_SPAN * LW0
    for _ in range(5):
        fs = centre + np.linspace(-span, span, pts)
        spacing = fs[1] - fs[0]
        dec = _dec_for(spacing)
        nsamp = SWEEP_N if dec <= 12 else SWEEP_N // 2
        p  = np.empty(pts); ch = np.empty(pts)
        for k, fq in enumerate(fs):
            z = capture(fq, dec, nsamp)
            p[k]  = float(np.var(z))
            ch[k] = coherence(z)
        coh   = float(ch[int(np.argmax(p))])
        base  = float(np.min(p))
        share = (p.max() - base) / max(float(np.sum(p - base)), 1e-300)
        contrast = p.max() / max(float(np.median(p)), 1e-300)
        if contrast < 2.0:
            span = min(span * 3.0, 8.0 * LW0)
            continue
        if share > 0.6:
            centre = float(fs[int(np.argmax(p))])
            span   = max(span / 5.0, 5e2)
            continue
        try:
            f0, lw, p0, pbase = fit_lorentz3(fs, p)
        except ValueError:
            centre = float(fs[int(np.argmax(p))])
            span   = max(span / 5.0, 5e2)
            continue
        inside = bool(fs[0] < f0 < fs[-1])
        # The baseline must be added back before judging the fit: comparing
        # data-with-floor against a floor-free Lorentzian rejects a CORRECT fit
        # whenever the floor matters, which is exactly the broad-line case the
        # baseline removal was added for.
        fitc   = p0 / (1.0 + (2.0 * (fs - f0) / lw) ** 2) + pbase
        resid  = float(np.sqrt(np.mean(((p - fitc) / p.max()) ** 2)))
        if inside and 1.5 * spacing <= lw <= (fs[-1] - fs[0]) and resid < 0.15:
            area = (np.pi / 2.0) * p0 * lw
            lw_t, bracketed = _deconv_lw(lw, f0, fs, dec)
            if bracketed:
                return np.nan, np.nan, np.nan, np.nan, coh
            return f0, lw_t, p0, area, coh
        if inside and spacing / 5.0 < lw < 1.5 * spacing:
            centre, span = f0, max(2.0 * lw, 5e2)
        else:
            centre = float(fs[int(np.argmax(p))])
            span   = max(span / 5.0, 5e2)
    return np.nan, np.nan, np.nan, np.nan, np.nan        # honest failure


reso.noise_tank(TANK_SH); reso.noise_out(None)
reso.clear(); time.sleep(0.05)
F_OL, LW_OL, PK_OL, AREA_OL, COH_OL = noise_sweep(reso.f0_q)
assert np.isfinite(LW_OL), 'open-loop sweep unresolved - something is badly wrong'
Q_OL = F_OL / LW_OL
print('open loop: f0 %.1f Hz, linewidth %.0f Hz, Q %.1f  (known: %.1f)'
      % (F_OL, LW_OL, Q_OL, reso.q_q))
print('           coherence %.2f (want ~1 - pure noise, no tone)' % COH_OL)
if EMULATION:
    assert abs(Q_OL / reso.q_q - 1.0) < 0.10, 'open-loop Q wrong - fix before scanning'
print('STEP 0: ready')

---
## Step 1 — Telling an oscillation from noise

The original test removes the drive and asks whether the signal survives. The replacement asks
a different question of the same record: **how tone-like is it?**

`coherence()` is the spread of the instantaneous power, divided by its average squared. For
pure noise that is 1. For a pure tone it is 0. In between it is exactly `1 − f²`, where `f` is
the fraction of the power the tone carries — so `tone_fraction()` reads `f` straight back out.

This measure was set aside earlier for spotting *mild* squashing, where it had to resolve a
10% change against roughly 30% scatter. Telling an oscillation from thermal noise is the easy
version of the same question: the two answers are near 0 and near 1, not 10% apart.

The cell below runs **both** tests at every point of a gain × phase grid and compares them
point by point.

In [ ]:
# --- the original test: remove the drive, see what survives -----------------
def sustain_ratio(A, phi):
    """Power left with the noise drive removed, as a fraction of driven power.
    A resonator decays to nothing; an oscillator holds itself up. Emulator only."""
    ps.set(A, phi); reso.clear(); time.sleep(0.010)
    if reso.clipped():
        ps.off(); reso.clear()
        return 1.0
    p_on = float(np.var(capture(F_OL, 10, 512)))
    reso.noise_tank(None)
    p_off = float(np.var(capture(F_OL, 10, 512)))
    reso.noise_tank(TANK_SH)
    ps.off(); reso.clear()
    return p_off / max(p_on, 1e-300)


# --- the replacement: how tone-like is the signal? --------------------------
def tone_reading(A, phi):
    """Tone fraction of the driven record. Works on any coil, real or not."""
    ps.set(A, phi); reso.clear(); time.sleep(0.010)
    z = capture(F_OL, 10, 2048)
    ps.off(); reso.clear()
    return tone_fraction(coherence(z) / max(COH_OL, 1e-300))


GAINS = 10.0 ** (GAINS_DB / 20.0)
NG, NP = len(GAINS), len(PHIS)
SUS  = np.full((NG, NP), np.nan)     # original reading
TONE = np.full((NG, NP), np.nan)     # replacement reading

t0 = time.time()
for i, A in enumerate(GAINS):
    for j, phi in enumerate(PHIS):
        SUS[i, j]  = sustain_ratio(A, phi)
        TONE[i, j] = tone_reading(A, phi)
    print('  gain %+6.1f dB done  (%d/%d rows, %.0f s)'
          % (GAINS_DB[i], i + 1, NG, time.time() - t0))
ps.off(); reso.clear()

# --- where does the replacement separate the two populations? ---------------
#  The original calls a point oscillating at SUS_OSC. Take that as truth, then
#  find the tone threshold that reproduces it, and check the two populations
#  are actually far apart rather than merely separable by a lucky cut.
truth = SUS >= SUS_OSC
osc_t, quiet_t = TONE[truth], TONE[~truth]
print()
print('original test  : %d of %d points oscillating' % (truth.sum(), truth.size))
if truth.sum() == 0 or (~truth).sum() == 0:
    print('FAIL - the grid contains only one population, so nothing can be')
    print('       calibrated. Widen GAINS_DB so it spans the boundary.')
    ok_sep = False
    TONE_OSC = np.nan
else:
    print('tone fraction  : oscillating %.3f to %.3f, quiet %.3f to %.3f'
          % (osc_t.min(), osc_t.max(), quiet_t.min(), quiet_t.max()))
    gap = osc_t.min() - quiet_t.max()
    TONE_OSC = 0.5 * (osc_t.min() + quiet_t.max())
    print('gap between them: %.3f      suggested threshold: %.3f' % (gap, TONE_OSC))
    ok_sep = gap > 0.15

    mism = np.where((TONE >= TONE_OSC) != truth)
    print()
    if len(mism[0]) == 0:
        print('every point classified identically by both tests')
    else:
        print('%d point(s) classified differently:' % len(mism[0]))
        for i, j in zip(*mism):
            print('   gain %+6.1f dB, phase %3.0f deg: original %s, replacement %s'
                  % (GAINS_DB[i], PHIS[j],
                     'osc ' if truth[i, j] else 'quiet',
                     'osc ' if TONE[i, j] >= TONE_OSC else 'quiet'))
    ok_sep = ok_sep and len(mism[0]) == 0

print()
print('STEP 1:', 'PASS' if ok_sep else
      'FAIL - the replacement does not reproduce the original classification')

---
## Step 2 — Is the loop still linear here?

The original test changes the coil's own thermal noise and checks the resonance shrinks in
proportion. The replacement changes something you *can* control on a real coil: it injects a
tone through the generator at two amplitudes 12 dB apart and requires the measured response
to grow in proportion.

Both are asking the same question — does twice the input give twice the output — but only one
of them needs a knob on the coil's temperature.

In [ ]:
# --- the original: change the coil's own noise ------------------------------
def linear_by_noise(A, phi):
    """Area ratio when the tank drive is dropped DRIVE_STEP shifts. Emulator only."""
    ps.set(A, phi); reso.clear()
    reso.noise_tank(TANK_SH);              time.sleep(0.02)
    a_hi = noise_sweep(reso.f0_q)[3]
    reso.noise_tank(TANK_SH + DRIVE_STEP); time.sleep(0.02)
    a_lo = noise_sweep(reso.f0_q)[3]
    reso.noise_tank(TANK_SH); ps.off(); reso.clear()
    if not (np.isfinite(a_hi) and np.isfinite(a_lo)):
        return np.nan
    return (a_hi / a_lo) / (4.0 ** DRIVE_STEP)      # 1.0 when perfectly linear


# --- the replacement: change a tone you control -----------------------------
def linear_by_tone(A, phi, drive_hi=0.20):
    """Response ratio for a tone at two amplitudes 12 dB apart. Works anywhere.

    The tone sits ON the resonance, so it probes the same part of the loop the
    thermal noise does. Amplitude is read from the register voltmeter, which is
    quick and needs no memory capture."""
    ps.set(A, phi); reso.clear()
    gen.freq(reso.f0_q); acq.tune(reso.f0_q)
    gen.amp(drive_hi);        time.sleep(SETTLE)
    a_hi = abs(acq.measure(128) - z_bg)
    gen.amp(drive_hi / 4.0);  time.sleep(SETTLE)
    a_lo = abs(acq.measure(128) - z_bg)
    gen.amp(0.0); ps.off(); reso.clear()
    if a_lo <= 0:
        return np.nan
    return (a_hi / a_lo) / 4.0                      # 1.0 when perfectly linear


# Walk up the strongest column, where compression appears first.
#  Pick the phase whose quiet points get closest to oscillating: that column
#  reaches the highest amplitudes, so compression shows up there first.
quiet = np.where(SUS < SUS_OSC, TONE, np.nan)
j_probe = int(np.nanargmax(np.nanmax(quiet, axis=0)))
print('probing along phase %.0f deg' % PHIS[j_probe])
print()
print('%9s %14s %14s' % ('gain [dB]', 'noise method', 'tone method'))
rows = []
for i in range(NG):
    if SUS[i, j_probe] >= SUS_OSC:
        continue                                    # oscillating: not a measurement
    r_noise = linear_by_noise(GAINS[i], PHIS[j_probe])
    r_tone  = linear_by_tone(GAINS[i], PHIS[j_probe])
    rows.append((GAINS_DB[i], r_noise, r_tone))
    print('%9.1f %14.3f %14.3f' % (GAINS_DB[i], r_noise, r_tone))

good = [(g, a, b) for g, a, b in rows if np.isfinite(a) and np.isfinite(b)]
if len(good) < 3:
    print()
    print('FAIL - too few usable points to compare the two methods')
    ok_lin = False
else:
    # Both should read 1.0 while linear and fall together as compression bites.
    da = np.array([abs(a - b) for _, a, b in good])
    print()
    print('largest disagreement between the two methods: %.3f' % da.max())
    lin_noise = [g for g, a, b in good if abs(a - 1.0) < 0.15]
    lin_tone  = [g for g, a, b in good if abs(b - 1.0) < 0.15]
    print('linear up to %+.1f dB by the noise method, %+.1f dB by the tone method'
          % (max(lin_noise) if lin_noise else float('nan'),
             max(lin_tone) if lin_tone else float('nan')))
    ok_lin = da.max() < 0.20
print()
print('STEP 2:', 'PASS' if ok_lin else
      'FAIL - the two linearity methods disagree; do not trust the replacement')

---
## Step 3 — What to carry to the real coil

The two numbers below are what this stage exists to produce. Write them down: on a real coil
they replace two decisions that can no longer be measured directly.

In [ ]:
print('=' * 70)
print(' CALIBRATED REPLACEMENTS - for use when the resonator is disabled')
print('=' * 70)
print()
if np.isfinite(TONE_OSC):
    print('  oscillation threshold : tone fraction >= %.3f' % TONE_OSC)
    print('      A point whose record is at least this tone-like is oscillating.')
    print('      Replaces: switching the coil noise off and seeing what survives.')
else:
    print('  oscillation threshold : NOT CALIBRATED - Step 1 did not separate')
print()
print('  linearity test        : tone at two amplitudes 12 dB apart, response')
print('                          ratio must stay within 15%% of 1.0')
print('      Replaces: dropping the coil noise 12 dB and checking the area.')
print()
print('  measured on           : Q = %.0f, linewidth %.0f Hz, coherence %.2f'
      % (Q_OL, LW_OL, COH_OL))
print()
print('  Both were checked against the originals on this emulated coil. Neither')
print('  can be re-checked once a real coil is attached, which is why this stage')
print('  runs first.')
print()
print('STEP 3: replacements calibrated' if np.isfinite(TONE_OSC) else
      'STEP 3: NOT calibrated - fix Step 1 first')

---
## Stage 12 summary

| Step | What it delivered |
|---|---|
| 1 | A tone-based oscillation test, checked point by point against the original across the whole grid |
| 2 | A linearity test driven by an injected tone rather than by the coil's own temperature |
| 3 | The two thresholds the real coil will use, and the evidence behind them |

Stage 10 can now run against a real coil by swapping in these two tests. Nothing else in it
depends on being able to change the plant's noise.

In [ ]:
# Board back to power-on behaviour.
reso.noise_tank(None); reso.noise_out(None)
ps.off()
gen.stop()
reso.disable()
acq.w(0x00, 0)
print('all quiet: feedback open, generator stopped, resonator disabled')